# Week 08 - Day 04 | Context Managers

**Topics:** `with` statement · `__enter__` / `__exit__` · Exception handling · `@contextlib.contextmanager` · `contextlib.suppress`

---

## Context Manager Cheatsheet

| Pattern | Syntax | Notes |
|---|---|---|
| Class-based | `__enter__` / `__exit__` | `__enter__` return value becomes the `as` variable |
| Suppress exception | `return True` in `__exit__` | `False`/`None` lets the exception propagate |
| Generator-based | `@contextlib.contextmanager` | Code before `yield` = enter, after = exit |
| Built-in suppress | `contextlib.suppress(ExcType)` | Silently ignores a given exception type |
| Multiple managers | `with a(), b():` | Entered left-to-right, exited right-to-left |

In [ ]:
# 1. Class-based context manager — Timer
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed * 1000:.2f}ms")
        return False    # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))

print(f"Sum: {total}")

In [ ]:
# 2. Suppressing exceptions in __exit__

class SuppressErrors:
    def __init__(self, exc_type):
        self.exc_type = exc_type

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exc_type):
            print(f"Suppressed: {exc_type.__name__}: {exc_val}")
            return True     # suppress
        return False        # propagate everything else

with SuppressErrors(ZeroDivisionError):
    print(1 / 0)

print("Program continues normally")

In [ ]:
# 3. Generator-based context manager
import contextlib
import time

@contextlib.contextmanager
def timer_cm():
    start = time.perf_counter()
    try:
        yield start             # becomes the 'as' value
    finally:
        elapsed = time.perf_counter() - start
        print(f"timer_cm elapsed: {elapsed * 1000:.2f}ms")

with timer_cm():
    total = sum(range(500_000))
print(f"Sum: {total}")

@contextlib.contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    resource = {"name": name, "open": True}
    try:
        yield resource
    finally:
        resource["open"] = False
        print(f"Releasing {name}")

with managed_resource("database") as db:
    print(f"Using {db['name']}, open={db['open']}")

In [ ]:
# 4. contextlib.suppress + multiple context managers
import contextlib
import os

with contextlib.suppress(FileNotFoundError):
    os.remove("does_not_exist.txt")
print("Still running after suppress")

@contextlib.contextmanager
def section(name):
    print(f"--- enter {name} ---")
    yield
    print(f"--- exit {name} ---")

with section("A"), section("B"):
    print("inside both sections")

In [ ]:
# 5. Practical: redirecting stdout
import contextlib
import io

buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    print("this goes into the buffer, not the terminal")

captured = buffer.getvalue()
print(f"Captured: {captured!r}")

## Summary

| Concept | Key Point |
|---|---|
| `with` statement | Guarantees `__exit__`/cleanup runs even on exceptions |
| `__enter__` | Sets up the resource; return value becomes the `as` variable |
| `__exit__` | Always runs; receives exception info; `True` suppresses it |
| `@contextlib.contextmanager` | Turns a generator into a context manager — before/after `yield` |
| `contextlib.suppress` | Shortcut for silently ignoring a given exception type |
| Multiple managers | `with a(), b():` enters left-to-right, exits right-to-left |

> **Next:** Day 05 — Type Hints (`typing` module, `Optional`, `Union`, generics)